## **Fresh Eyes on CMIP Data Workshop 2026**: Application with Julia

#### **Setup**
1. Download Julia from https://github.com/JuliaLang/juliaup#installation
2. Install Julia extension in Visual Code Studio market place
3. Code can be execute in 
    - Julia Read-Eval-Print-Loop (Julia: Start REPL)
    - Jupyter Notebook
    - ...
4. When selecting the Kernel in Jupyter:
    - Julia release channel (automatically uses most recent version of Julia on your machine)


In [1]:
print("Hello World")

Hello World

#### **Package management**
- For different projects you often need different additional packages or libraries
- In Julia the `pkg` helps you to manage libraries and make your project reproducible for others
    - You have a global environment in `~/.julia/environments/v1.13/` (in case of version 1.13)
    - You should create a local environment in your project folder
    - All packages are installed into the same directory `~/.julia/packages/`, independent of the environment
- The environent contains two files
    - `Project.toml`
    - `Manifest.toml`

In [ ]:
using Pkg # tell Julia to use pkg manager
Pkg.activate() # activate the global environment (should already be the case)
Base.active_project() # show which Project.toml file is currently active

  Activating project at `~/.julia/environments/v1.13`


"/Users/tim/.julia/environments/v1.13/Project.toml"

In [3]:
print(Base.load_path()) # shows all paths to environments that are currently checked when loading packages
print("\n") # print a linebreak
print(LOAD_PATH) # print LOAD_PATH variable, showing all environments which will currently be checked

["/Users/tim/.julia/environments/v1.13/Project.toml", "/Users/tim/.julia/juliaup/julia-1.13.1+0.aarch64.apple.darwin14/Julia-1.13.app/Contents/Resources/julia/share/julia/stdlib/v1.13"]
["@", "@v#.#", "@stdlib"]

In [4]:
Pkg.add("BenchmarkTools") # install BenchmarkTools package to global environment

     Project No packages added to or removed from `~/.julia/environments/v1.13/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.13/Manifest.toml`


In [5]:
Pkg.activate(".") # activate current directory as project environment
using BenchmarkTools # install BenchmarkTools (global environment will be used)

  Activating project at `~/Documents/GitHub/Julia-CMIP`


In [18]:
deleteat!(LOAD_PATH, 2) # remove the global environment from LOAD_PATH so that BenchmarkTools is not found when using it in this notebook
using BenchmarkTools # this should now throw an error since BenchmarkTools is not installed in the current environment

ArgumentError: ArgumentError: Package BenchmarkTools not found in current path, maybe you meant `import/using .BenchmarkTools`.
- Otherwise, run `import Pkg; Pkg.add("BenchmarkTools")` to install the BenchmarkTools package.

In [6]:
Pkg.add("BenchmarkTools") # install BenchmarkTools package to local environment (create Project.toml and Manifest.toml files)
using BenchmarkTools

     Project No packages added to or removed from `~/Documents/GitHub/Julia-CMIP/Project.toml`
    Manifest No packages added to or removed from `~/Documents/GitHub/Julia-CMIP/Manifest.toml`


In [7]:
# Aftere cloning a repository, you run Pkg.instantiate() to install all dependencies listed in the Project.toml 
# and Manifest.toml files. This will ensure that your local environment has all the necessary packages to run the 
# code in the repository.
Pkg.instantiate()

#### **Why (and when) Julia?**

- Both python and julia are high level languages (human readible) while processors can only read machine code
- Both languages use dynamics typing, so variables types do not need to be defiend beforehand
- However, they differ in the way how code is translated to machine code
    - **Python** is an interpreter languge compiles code into intermediate bytecode (saved in `__pycache__`), which the interpreter then executes step-by-step at runtime without compiling it into native CPU machine code.
    - **Julia** uses JIT (just in time) compilation, where a function is compiled just before it is called (possible through multiple dispatch)
    - In **C** the whole program code is compiled before execution
- Python has functions that are precompiled in **C**
    - If you use only these functions, python will not be slower
    - If you use a lot of custom functions with loops and conditionals, they will be faster in Julia
- Generally
    - Use python for deep learning, tabular data processing (pandas) or general software engineering (object oriented)
    - Use julia for scientific coding (differential equations, ABMs, physical models, ...) and mathematics and optimisation

#### Example
Calculate the eucledian distance on two vectors with 10 Mio elements ($d=\sqrt{\sum(a_i-b_i)^2}$).
1. Using native python
2. Using numpy (precompiled functions in C)
3. Using native julia

In [ ]:
using Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

using BenchmarkTools
using PythonCall  # Package that allows to call Python from Julia (loads numpy library as specified in CondaPkg.toml)

np = pyimport("numpy") # import numpy library from Python

  Activating project at `~/Documents/GitHub/Julia-CMIP`


Python: <module 'numpy' from '/Users/tim/Documents/GitHub/Julia-CMIP/.CondaPkg/.pixi/envs/default/lib/python3.12/site-packages/numpy/__init__.py'>

In [ ]:
# Native Python Function (Runs in Python's bytecode interpreter)
@pyexec """
def euclidean_native_py(x, y):
    total = 0.0
    for i in range(len(x)):
        diff = x[i] - y[i]
        total += diff * diff
    return total ** 0.5
""" => euclidean_native_py

# Native Julia Function (Plain loop, JIT-compiled to native CPU machine code)
function euclidean_native_jl(x, y)
    total = 0.0
    for i in eachindex(x, y)
        diff = x[i] - y[i]
        total += diff * diff
    end
    return sqrt(total)
end

# Native Julia Function (Plain loop, JIT-compiled to native CPU machine code)
function euclidean_native_jl_fast(x, y)
    total = 0.0
    # @ is a macro that optimizes the compilation of the loop
    # @inbound tells the compiler to skip bounds checking for array indexing, which can improve performance
    # @simd, Single Instruction, Multiple Data, lets the compiler parallelize the operation
    @inbounds @simd for i in eachindex(x, y)
        diff = x[i] - y[i]
        total += diff * diff
    end
    return sqrt(total)
end

euclidean_native_jl_fast (generic function with 1 method)

In [ ]:
# create random array of length N with Float64 values between 0 and 1
N = 10_000_000
a_jl = rand(Float64, N)
b_jl = rand(Float64, N)

# Convert data into Python list and NumPy array formats
a_pylist = pylist(a_jl);       b_pylist = pylist(b_jl);
a_numpy  = np.array(a_jl);     b_numpy  = np.array(b_jl);

# @btime is a macro from BenchmarkTools.jl that measures the execution time of a function call
println("1. Native Python (Interpreted for-loop):")
@btime $euclidean_native_py($a_pylist, $b_pylist)

println("\n2. Python + NumPy (Pre-compiled C library):")
@btime $np.linalg.norm($a_numpy - $b_numpy)

println("\n3. Native Julia (Pure Julia for-loop, JIT compiled):")
@btime euclidean_native_jl($a_jl, $b_jl)

println("\n4. Native Julia (Optimized for-loop, JIT compiled):")
@btime euclidean_native_jl_fast($a_jl, $b_jl);

1. Native Python (Interpreted for-loop):
  377.426 ms (2 allocations: 32 bytes)

2. Python + NumPy (Pre-compiled C library):
  11.788 ms (9 allocations: 176 bytes)

3. Native Julia (Pure Julia for-loop, JIT compiled):
  9.373 ms (0 allocations: 0 bytes)

3. Native Julia (Pure Julia for-loop, JIT compiled):
  9.373 ms (0 allocations: 0 bytes)

4. Native Julia (Optimized for-loop, JIT compiled):
  2.595 ms (0 allocations: 0 bytes)


In [ ]:
# Capture the outputs
# Convert Python objects (Py) to native Julia Float64 using pyconvert
res_py    = pyconvert(Float64, euclidean_native_py(a_pylist, b_pylist))
res_numpy = pyconvert(Float64, np.linalg.norm(a_numpy - b_numpy))
res_jl    = euclidean_native_jl(a_jl, b_jl)
res_jl_fast = euclidean_native_jl_fast(a_jl, b_jl)

# Print the exact values and their differences
println("Native Python: ", res_py)
println("Python NumPy:  ", res_numpy)
println("Native Julia:  ", res_jl)
println("Julia (Optmz): ", res_jl_fast)

# Check approximate equality (using isapprox or the ≈ symbol)
println("\nAre all four approximately equal? ", res_py ≈ res_numpy ≈ res_jl ≈ res_jl_fast)
println("Are Python and Julia strictly identical (==)? ", res_py == res_jl)

Native Python: 1290.8442164820844
Python NumPy:  1290.8442164820888
Native Julia:  1290.8442164820844
Julia (Optmz): 1290.8442164820879

Are all four approximately equal? true
Are Python and Julia strictly identical (==)? true


**Things to Notice**
1. **Native Python is slowest**: As expected, the pure Python loop takes the longest because the CPython interpreter has to check variable types and create new float objects in memory on every single iteration.
2. **Native Julia is even faster than NumPy**: even though NumPy also used pre-compiled C code, it is slower than Julia, becuase it evaluates the function `np.linalg.norm(a-b)` in two steps. First it runs the $a-b$ statement and writes it temporarily into RAM (9 allocations), then it runs the NumPy function. This extra step is what makes it slower than Julia which executes everything as a single pass without allocating temporary arrays. 
3. **@inbound and @simd**: both optimization speed up the Julia implementation 5 times compared to the basic Julia implementation without macros.
4. **Warm-Up**: The very first time you run a Julia function, there is a brief pause while Julia's JIT compiler translates the code into native machine instructions. The @btime macro excludes this one-time compilation overhead by executing an untimed warm-up run first and then reporting the minimum execution time across multiple runs.
5. **Precision**: A basic sequential loop in Python and a basic sequential loop in Julia produce bit-for-bit identical results (==) because they add the numbers in the exact same left-to-right order. In contrast, NumPy and the @simd-optimized Julia loop use SIMD vectorization (accumulating multiple partial sums in parallel inside CPU registers). Changing the order of addition slightly alters the 13th–15th decimal places, and accumulating into separate partial sums actually reduces floating-point rounding error.


In [ ]:
# Use PythonCall to import Pythons xarray and numpy
xr = pyimport("xarray")
np = pyimport("numpy");

In [ ]:
path = "." # put path to file on your machine here
file = "tas_Amon_ACCESS-CM2_historical_r1i1p1f1_gn_185001-201412.nc"
filepath = joinpath(path, file)

# Load in Julia and convert to concrete Float32 (since tas has no missing values)
# Rasters.checkmem!(false) # Turn off the macOS free-RAM check globally for the session
tas_raw_jl = Raster(filepath, name=:tas) # load file and select tas variable

In [ ]:
# modiy is a function of Rasters.jl that modifes the values type while keeping the dimensions and metadata
tas_clean_jl = modify(Array{Float32}, tas_raw_jl);

In [ ]:
# Load in Python via xarray (.load() forces it into RAM for a fair benchmark
ds_py = xr.open_dataset(filepath).load()
tas_raw_py = ds_py["tas"]; # select the tas variable from the xarray dataset

In [ ]:
function calc_anom_native_jl(tas_raw)
    # Convert to °C
    tas = tas_raw .- 273.15f0
    
    # Slice reference (360 months) and analysis (432 months) periods
    # @view is a macro that specifies that the slice is not copied into new memory
    # we can do this here because we do not modify the reference period
    tas_ref = @view tas[Ti(DateTime(1981, 1, 1) .. DateTime(2011, 1, 1))]
    tas_period = tas[Ti(DateTime(1979, 1, 1) .. DateTime(2015, 1, 1))]
    
    nx, ny, nt_ref = size(tas_ref) # return dimensions of reference period (X,Y,Ti)
    nt_per = size(tas_period, 3) # number of time steps in analysis period (432 months)
    
    # Compute average temperature for every month in reference period (192 × 144 × 12)
    clim = zeros(Float32, nx, ny, 12) # create array of 0s to hold monthly averages
    n_years_ref = nt_ref ÷ 12 # number of years in the reference period (÷ is integer division, like python //, type \dif and tab)
    
    # @inbound is a macro that tells Julia to skip some securtity checks when processing an array (speed up)
    @inbounds for m in 1:12
        for k in 0:(n_years_ref - 1)
            t = m + k * 12 # compute time intex for current month m and year k (january first year = 1, january second year = 13, etc.)
            # @simd  tells the processor to process multiple values in parallel (speed up)
            @simd for i in 1:(nx * ny)
                # The array is saved as a linear array in memory, so we also index it in this way
                # For each grid cell and month m in clim, we add the corresponding grid value of each month in tas_ref, indexed by t
                # (m - 1) * nx * ny calculates the offset in the linear array
                clim[i + (m - 1) * nx * ny] += tas_ref.data[i + (t - 1) * nx * ny]
            end
        end
    end
    clim ./= n_years_ref
    
    # Subtract monthly climatology in-place
    tas_anom = copy(tas_period)
    @inbounds for t in 1:nt_per
        m = mod1(t, 12)
        for y in 1:ny
            @simd for x in 1:nx
                tas_anom.data[x, y, t] -= clim[x, y, m]
            end
        end
    end
    
    return tas_anom
end;

In [ ]:
function calc_anom_native_jl(tas_raw)
    # Convert to °C
    tas = tas_raw .- 273.15f0
    
    # Slice reference (360 months) and analysis (432 months) periods
    # @view is a macro that specifies that the slice is not copied into new memory
    # we can do this here because we do not modify the reference period
    tas_ref = @view tas[Ti(DateTime(1981, 1, 1) .. DateTime(2011, 1, 1))]
    tas_anom = tas[Ti(DateTime(1979, 1, 1) .. DateTime(2015, 1, 1))]
    
    nx, ny, nt_ref = size(tas_ref) # return dimensions of reference period (X,Y,Ti)
    nt_per = size(tas_anom, 3) # number of time steps in analysis period (432 months)
    
    # Compute average temperature for every month in reference period (192 × 144 × 12)
    clim = zeros(Float32, nx, ny, 12) # create array of 0s to hold monthly averages
    n_years_ref = nt_ref ÷ 12 # number of years in the reference period (÷ is integer division, like python //, type \dif and tab)
    
    # @inbound is a macro that tells Julia to skip some securtity checks when processing an array (speed up)
    @inbounds for m in 1:12
        for k in 0:(n_years_ref - 1)
            t = m + k * 12 # compute time intex for current month m and year k (january first year = 1, january second year = 13, etc.)
            for y in 1:ny
                # @simd  tells the processor to process multiple values in parallel (speed up)
                @simd for x in 1:nx
                    # sum the grid values for each month of every year in the ref period
                    clim[x, y, m] += tas_ref.data[x, y, t]
                end
            end
        end
    end

    # calculate the monthly average. 
    clim ./= n_years_ref # the . operator ensures that division is applied element wise
    
    # Subtract monthly averages from analysis period
    @inbounds for t in 1:nt_per
        m = mod1(t, 12) # modulo function that returns month index, for perfect multiple returns 12 instead of 0 (mod1(12, 12) = 12))
        for y in 1:ny
            @simd for x in 1:nx
                tas_anom.data[x, y, t] -= clim[x, y, m]
            end
        end
    end
    
    return tas_anom
end;

In [ ]:
# here we implement the same algorithm using first native Python loops and then xarray to show the key differences
# in performance between Julia and Python!

@pyexec """
def calc_anom_native_py(tas_da):
    import numpy as np  # <-- Put the import INSIDE the function!
    
    # Slice periods using xarray, then extract raw 3D numpy arrays (time, lat, lon)
    ref_arr = (tas_da.sel(time=slice('1981-01-01', '2010-12-31')).values - 273.15)
    per_arr = (tas_da.sel(time=slice('1979-01-01', '2014-12-31')).values - 273.15)
    
    nt_ref, ny, nx = ref_arr.shape
    nt_per = per_arr.shape[0]
    
    # Compute 12 monthly climatologies using native Python loops
    clim = np.zeros((12, ny, nx), dtype=np.float32)
    for m in range(12):
        for y in range(ny):
            for x in range(nx):
                total = 0.0
                count = 0
                for t in range(m, nt_ref, 12):
                    total += ref_arr[t, y, x]
                    count += 1
                clim[m, y, x] = total / count
                
    # Subtract monthly climatology from 1979-2014 period
    anom = np.empty_like(per_arr)
    for t in range(nt_per):
        m = t % 12
        for y in range(ny):
            for x in range(nx):
                anom[t, y, x] = per_arr[t, y, x] - clim[m, y, x]
                
    return anom
""" => calc_anom_native_py

@pyexec """
def calc_anom_xarray_py(tas_da):
    tas = tas_da - 273.15
    tas_ref = tas.sel(time=slice('1981-01-01', '2010-12-31'))
    tas_clim = tas_ref.groupby('time.month').mean('time')
    
    # Optimization: slice 1979-2014 BEFORE groupby subtraction so we don't 
    # waste time computing anomalies for 1850-1978
    tas_period = tas.sel(time=slice('1979-01-01', '2014-12-31'))
    tas_anom = tas_period.groupby('time.month') - tas_clim
    return tas_anom.values
""" => calc_anom_xarray_py

In [ ]:
# Benchmarking
println("1. Native Python (Interpreted for-loops):")
# We use @time (1 run) instead of @btime here because Python loops take ~5-8 seconds per run!
@time res_py_loop = calc_anom_native_py(tas_raw_py);

println("\n2. Python Fast Operation (Vectorized xarray groupby):")
@btime calc_anom_xarray_py($tas_raw_py);

println("\n3. Native Julia (JIT-compiled for-loops):")
@btime calc_anom_native_jl($tas_clean_jl);

# Validate outputs are approximately equal
res_xr_np = pyconvert(Array{Float32, 3}, calc_anom_xarray_py(tas_raw_py))
res_py_np = pyconvert(Array{Float32, 3}, res_py_loop)
res_jl = calc_anom_native_jl(tas_clean_jl)
res_jl_array = Array(res_jl) # convert Rasters.jl Raster to a standard Julia array for comparison

# Note: Python arrays are (time, lat, lon) = (432, 144, 192), 
# while Julia Rasters are (lon, lat, time) = (192, 144, 432).
# We permute Python's dimensions to (3, 2, 1) to compare grid cell by grid cell:
res_xr_permuted = permutedims(res_xr_np, (3, 2, 1))
res_py_permuted = permutedims(res_py_np, (3, 2, 1))

println("\nAre all three results approximately equal (≈)? ", res_jl_array ≈ res_xr_permuted ≈ res_py_permuted)
println("Max absolute difference (Xarray vs Julia): ", maximum(abs.(res_jl_array .- res_xr_permuted)))

**Raster File Summary**
- First Section (`192×144×1980 Raster{Union{Missing, Float32}, 3} tas`): three dimensions, each value can either be float or Missing
    - The file acutally does not contain missing values
    - Doing the calculation with one variable type increases speed dramatically
- Second Section: Axis dimensions, longitude, latitude and time stamp
- Third Section: Metadata
- Fourth Section: Spatial and temporal extend and coordinate system

#### **Calculate temperature Anomaly**
We now calculate the temperature anomaly doing the following steps
1. Convert temperature from Kelvin to degree celsius
2. Define a reference period (1980-2010) and a analysis period (1979-2015)
3. Compute the average monthly temperature for the reference period
4.  Substract from analysis period
